# Rational design and structural characterization of bioactive molecules - UniMi 2025/2026

In this tutorial we will present you Boltz2 and its use, as well as guidelines on how to critically evaluate the quality of the generated predicted structures.

All data and information are available from this tutorial [GitHub page](https://github.com/LucaChiesa/UniMi_2024_AlphaFold_Tutorial).

##  Environment setup

In [ ]:
# @title Install dependencies { display-mode: "form" }
import os
print("Installing Boltz2")
if not os.path.isfile("BOLTZ_READY"):
    os.system("pip install py3Dmol")
    os.system("pip install boltz[cuda] -U")
    os.system("touch BOLTZ_READY")

if not os.path.isfile('DATA_READY'):
    os.system("git clone -b Boltz2 -q https://github.com/LucaChiesa/UniMi_2024_AlphaFold_Tutorial")
    os.system("touch DATA_READY")
print("Tutorial material downloaded")

import torch
if torch.cuda.is_available():
    print('Running on GPU')
else:
    print("WARNING: no GPU detected, will be using CPU")

In [ ]:
# @title Set local variables { display-mode: "form" }
github = 'UniMi_2024_AlphaFold_Tutorial/'
protein_structures = f'{github}protein_structures'
reference_structures = f'{github}reference_structures'
inputs = f'{github}inputs'

import warnings
warnings.filterwarnings('ignore')
from google.colab import files
from UniMi_2024_AlphaFold_Tutorial.utils import *

In [ ]:
# @title Javascript test { display-mode: "form" }
import py3Dmol
test_file = f"{protein_structures}/1ezg.pdb"
view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width = 400, height = 300)
view.addModel(open(test_file,'r').read(),'pdb')
view.setStyle({'model':0},{'cartoon': {'color':'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
# @title  Display the FASTA file  { display-mode: "form" }
import textwrap
with open(f"{inputs}/example_proteins.fasta") as fasta_inp:
    fasta_strs = fasta_inp.read().split()
for fasta_str in fasta_strs:
    print('\n'.join(textwrap.wrap(fasta_str)))

## Example system - Segment of a viral protein
Precalculated example of the results obtained using Boltz2

In [ ]:
# @title  Display the MSA  { display-mode: "form" }
#@markdown Display 10 sequences form the MSA
import matplotlib
import csv
msa = []
with open(f'{protein_structures}/boltz_results_example_protein/msa/example_protein_0.csv', newline='') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
    for i,row in enumerate(spamreader):
      if i == 0:
        continue
      msa.append(row[1])
for seq in msa[:500:50]:
    print(seq)

In [ ]:
# @title  Display MSA coverage  { display-mode: "form" }
%matplotlib inline
seq = msa[0]
msa = np.array([[s for s in seqs[:len(seq)]] for seqs in msa])
plot_msa_v2(msa)
plt.show()
plt.close()

In [ ]:
# @title Display 3D structure { display-mode: "form" }
#from litaf.utils import show_pdb
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

#@markdown Click on any atom to show a label with the residue name, the residue number, and the plDDT value.<br/>Click a second time on the same atom to hide the label.


vprot_path = "boltz_results_example_protein/predictions/example_protein/"

pdb_filename = f"{protein_structures}/{vprot_path}/example_protein_model_0.cif"

show_pdb(pdb_filename, 1, show_sidechains, show_mainchains, color, extension='cif').show()
if color == "lDDT":
    plot_plddt_legend().show()

In [ ]:
# @title Plot plDDT & PAE { display-mode: "form" }

plddt = np.load(f"{protein_structures}/{vprot_path}/plddt_example_protein_model_0.npz")

plddts_plot = plot_plddts([plddt['plddt']*100])

plddts_plot.savefig(f"{protein_structures}/{vprot_path}/plddts_plot.png")

In [ ]:
# @title Plot PAE& PDE { display-mode: "form" }

pae = np.load(f"{protein_structures}/{vprot_path}/pae_example_protein_model_0.npz")
pde = np.load(f"{protein_structures}/{vprot_path}/pde_example_protein_model_0.npz")

pae_plot = plot_paes([pae['pae']])
pde_plot = plot_paes([pde['pde']], plot_type='PDE')

pae_plot.savefig(f"{protein_structures}/{vprot_path}/pae_plot.png")
pae_plot.savefig(f"{protein_structures}/{vprot_path}/pde_plot.png")

In [ ]:
# @title Show confidence values { display-mode: "form" }
import json
import pprint

with open("/content/UniMi_2024_AlphaFold_Tutorial/protein_structures/boltz_results_example_protein/predictions/example_protein/confidence_example_protein_model_0.json") as inp:
  pprint.pprint(json.load(inp))

## SARS-CoV-2 3C-like protease
We run full calculations for this one, from using MMseqs2 to evaluating the complex quality

In [ ]:
# @title Input file { display-mode: "form" }

from pathlib import Path
import logging
import os
import yaml

sars_cov_path = 'boltz_results_sars-cov2_complex/predictions/sars-cov2_complex'

with open(f'{inputs}/sars-cov2_complex.yaml') as f:
    input_cont = f.read()
with open(f'{inputs}/sars-cov2_complex.yaml') as f:
    input_data = yaml.safe_load(f)

print(input_cont)

In [ ]:
# @title Display ligand structure { display-mode: "form" }

from rdkit import Chem
from rdkit.Chem import Draw
mol = Chem.MolFromSmiles(input_data['sequences'][1]['ligand']['smiles'])
Draw.MolToImage(mol)

In [ ]:
# @title Run predictions { display-mode: "form" }
os.system(f"boltz predict {inputs}/sars-cov2_complex.yaml --use_msa_server --out_dir {protein_structures}")

In [ ]:
# @title Display MSA coverage { display-mode: "form" }
%matplotlib inline
msa = []
with open(f'{protein_structures}/boltz_results_sars-cov2_complex/msa/sars-cov2_complex_0.csv', newline='') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
    for i,row in enumerate(spamreader):
      if i == 0:
        continue
      msa.append(row[1])
seq = msa[0]
msa = np.array([[s for s in seqs[:len(seq)]] for seqs in msa])
plot_msa_v2(msa)
plt.show()
plt.close()

In [ ]:
# @title Display 3D structure { display-mode: "form" }
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

pdb_filename = f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0.cif"

show_pdb(pdb_filename, 2, show_sidechains, show_mainchains, color, extension='cif').show()
if color == "lDDT":
    plot_plddt_legend().show()

In [ ]:
# @title Plot plDDT{ display-mode: "form" }

plddt = np.load(f"{protein_structures}/{sars_cov_path}/plddt_sars-cov2_complex_model_0.npz")

plddts_plot = plot_plddts([plddt['plddt']*100])

plddts_plot.savefig(f"{protein_structures}/{sars_cov_path}/plddts_plot.png")


In [ ]:
# @title Plot PAE & PDE { display-mode: "form" }

pae = np.load(f"{protein_structures}/{sars_cov_path}/pae_sars-cov2_complex_model_0.npz")
pde = np.load(f"{protein_structures}/{sars_cov_path}/pde_sars-cov2_complex_model_0.npz")

pae_plot = plot_paes([pae['pae']])
pde_plot = plot_paes([pde['pde']], plot_type='PDE')

pae_plot.savefig(f"{protein_structures}/{sars_cov_path}/pae_plot.png")
pae_plot.savefig(f"{protein_structures}/{sars_cov_path}/pde_plot.png")

In [ ]:
# @title Show confidence values { display-mode: "form" }

with open(f"{protein_structures}/{sars_cov_path}/confidence_sars-cov2_complex_model_0.json") as inp:
  pprint.pprint(json.load(inp))

In [ ]:
# @title Align AlphaFold models on reference structures { display-mode: "form" }
ref_b_file = f'{reference_structures}/6XQS.pdb'
ref_b_aligned = f'{reference_structures}/6XQS_aligned.cif'
ref_a_file = f'{reference_structures}/6WQF.pdb'
ref_a_aligned = f'{reference_structures}/6WQF_aligned.cif'

print("Align references")
align_on_ref(ref_a_file, ref_b_file, 'SV6', model_is_pdb=True, example=False)
align_on_ref(f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0.cif", ref_b_file, 'SV6', example = True)

In [ ]:
# @title Comparison between the free protein (PDBID: 6WQF) and the ligand bound protein (PDBID: 6XQS)  { display-mode: "form" }

holo_cartoon_color = "green" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
apo_cartoon_color = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
holo_sidechain_color = "cyan" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
apo_sidechain_color = "yellow" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

#@markdown Click on any atom to show a label with the residue name, the residue number, and the atom name.<br/>Click a second time on the same atom to hide the label.

show_aligned(ref_a_aligned, ref_b_aligned, ['SV6'],
             model_cartoon_color = apo_cartoon_color,
             ref_cartoon_color = holo_cartoon_color,
             ligand_colors = [ligand_color+'Carbon'],
             model_sidecahin_color = apo_sidechain_color+'Carbon',
             ref_sidecahin_color = holo_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(name_model = 'Apo', name_ref = 'Holo',
             model_cartoon_color = apo_cartoon_color,
             ref_cartoon_color = holo_cartoon_color,
             ligand_color = ligand_color,
             model_sidecahin_color = apo_sidechain_color,
             ref_sidecahin_color = holo_sidechain_color).show()

In [ ]:
# @title Comparison between the AlphaFold model and the free protein (PDBID: 6WQF) { display-mode: "form" }

reference_cartoon_color = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_cartoon_color = "magenta" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
reference_sidechain_color = "yellow" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_sidechain_color = "purple" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

aligned_model = f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0_aligned.cif"

show_aligned(aligned_model, ref_a_aligned, ['LIG1'],
             model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_colors = [ligand_color+'Carbon'],
             model_sidecahin_color = AF_model_sidechain_color+'Carbon',
             ref_sidecahin_color = reference_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color,
             model_sidecahin_color = AF_model_sidechain_color,
             ref_sidecahin_color = reference_sidechain_color).show()

In [ ]:
# @title Comparison between the AlphaFold model and the ligand bound protein (PDBID: 6XQS)  { display-mode: "form" }

reference_cartoon_color = "green" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_cartoon_color = "magenta" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color_model = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color_ref = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
reference_sidechain_color = "cyan" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_sidechain_color = "purple" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

aligned_model = f"{protein_structures}/{sars_cov_path}/sars-cov2_complex_model_0_aligned.cif"
show_aligned(aligned_model, ref_b_aligned, ['LIG1','SV6'],
             model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_colors = [ligand_color_model+'Carbon', ligand_color_ref+'Carbon'],
             model_sidecahin_color = AF_model_sidechain_color+'Carbon',
             ref_sidecahin_color = reference_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color_model,
             model_sidecahin_color = AF_model_sidechain_color,
             ref_sidecahin_color = reference_sidechain_color).show()

In [ ]:
# @title Comparison affinity predictions with experimental values  { display-mode: "form" }
#@markdown Boltz2 can predict the binding affinity of a co-colded molecule.

#@markdown Search in this [paper](https://pubs.acs.org/doi/full/10.1021/acsmedchemlett.4c00146) for the experimental affinity of the molecule and compare it to the prediction

#@markdown **IMPORTANT:** Boltz2 measures binding affinity in log(IC50) with IC50 experessed in μM. A molecule with IC50 of 1 μM will have a predicted value of 0.

import json
import pprint
with open(f"{protein_structures}/{sars_cov_path}/affinity_sars-cov2_complex.json") as inp:
  affinity = json.load(inp)
pprint.pprint(affinity)


In [ ]:
# A bit of help to calculate the logarithm

import math

math.log10(1e5)

In [ ]:
# @title Comparison between the AlphaFold model and the experimental (PDBID: 8U9K)  { display-mode: "form" }

reference_cartoon_color = "green" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_cartoon_color = "magenta" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color_model = "white" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
ligand_color_ref = "orange" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
reference_sidechain_color = "cyan" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
AF_model_sidechain_color = "purple" #@param ["green", "cyan", "magenta", "purple", "white", "orange", "yellow", "blue"]
height = 300 #@param {type:"number"}
width = 400 #@param {type:"number"}

ref_b_file = f'{reference_structures}/6XQS.pdb'
ref_b_aligned = f'{reference_structures}/6XQS_aligned.cif'
ref_a_file = f'{reference_structures}/8U9K.pdb'
ref_a_aligned = f'{reference_structures}/8U9K_aligned.cif'

print("Align references")
align_on_ref(ref_a_file, ref_b_file, 'SV6', model_is_pdb=True, example=False)

show_aligned(aligned_model, ref_a_aligned, ['LIG1','W0B'],
             model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_colors = [ligand_color_model+'Carbon', ligand_color_ref+'Carbon'],
             model_sidecahin_color = AF_model_sidechain_color+'Carbon',
             ref_sidecahin_color = reference_sidechain_color+'Carbon',
             width=width,
             height=height).show()
plot_chain_legend(model_cartoon_color = AF_model_cartoon_color,
             ref_cartoon_color = reference_cartoon_color,
             ligand_color = ligand_color_model,
             model_sidecahin_color = AF_model_sidechain_color,
             ref_sidecahin_color = reference_sidechain_color).show()